In [1]:
from datetime import datetime, timedelta, date
import requests
import time
import pandas as pd
import holidays
from category_encoders import TargetEncoder
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
import xgboost as xgb
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Pipeline opstellen

## Ophalen data

In [7]:
def ophalenKijkcijferData(startDate, endDate):
  print(f"Ophalen kijkcijfer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
  kijkcijfersData = []
  #elke dag ophalen (startDate is huidige dag)
  while startDate <= endDate:
    datum = f"{startDate.year}-{startDate.month}-{startDate.day}"
    url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"

    try:
      response = requests.get(url)
      if response.status_code == 200:
        data = response.json()
        programmaLijst = data.get('hydra:member', [])
                        
        for programma in programmaLijst:
          try:
            kijkcijfersData.append({
              'dateDiff': programma.get('dateDiff'),
              'ranking': programma.get('ranking'),
              'description': programma.get('description'),
              'channel': programma.get('channel'),
              'startTime': programma.get('startTime'),
              'rLength': programma.get('rLength'),
              'rateInK': programma.get('rateInK'),
              'live': programma.get('live')
            })
                                   
          except Exception as e:
            print(f"error {datum}: {e}")         
      else:
        print(f"no data {datum}")
                        
    except Exception as e:
      print(f"error: {e}")
    
    startDate += timedelta(days=1)

  print("KijkcijferData opgehaald")
  df = pd.DataFrame(kijkcijfersData)

  return df

def ophalenWeerData(startDate, endDate):
    latitude = 51.05
    longitude = 3.7167
    today = datetime.today().date()

    hourly_vars = [
        "temperature_2m", "apparent_temperature", "weather_code", "precipitation",
        "rain", "snowfall", "cloud_cover", "windspeed_10m", "sunshine_duration"
    ]
    
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }

    def fetch_weather_data(api_url, start, end):
        params = common_params.copy()
        params.update({
            "start_date": start.strftime('%Y-%m-%d'),
            "end_date": end.strftime('%Y-%m-%d')
        })
        print(f"Ophalen weer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
        response = requests.get(api_url, params=params)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df = pd.DataFrame({var: data.get(var, []) for var in hourly_vars})
            df["timestamp"] = pd.to_datetime(data.get("time", []))
            if not df.empty:
                df["hour"] = df["timestamp"].dt.hour
                df["day_of_week"] = df["timestamp"].dt.dayofweek
                df["month"] = df["timestamp"].dt.month
                df["year"] = df["timestamp"].dt.year
            return df
        else:
            print(f"Fout bij ophalen data: {response.status_code}")
            print(response.text)
            return pd.DataFrame()

    dataframes = []

    # Historische data
    if startDate.date() < today:
        hist_end = min(endDate.date(), today - timedelta(days=1))
        dataframes.append(fetch_weather_data(
            "https://archive-api.open-meteo.com/v1/archive",
            startDate, datetime.combine(hist_end, datetime.min.time())
        ))

    # Forecast data
    if endDate.date() >= today:
        forecast_start = max(startDate, datetime.combine(today, datetime.min.time()))
        dataframes.append(fetch_weather_data(
            "https://api.open-meteo.com/v1/forecast",
            forecast_start, endDate
        ))

    if dataframes:
        print("Weerdata opgehaald")
        return pd.concat(dataframes).sort_values("timestamp").reset_index(drop=True)
    else:
        return pd.DataFrame()


In [6]:
teVoorspellen = pd.DataFrame({
  'dateDiff' : ['2025-05-13T00:00:00.000000', '2025-05-13T00:00:00.000000'],
  'ranking': ['13', '234'],
  'description': ['THUIS', 'HET 7 UUR-JOURNAAL'],
  'channel': ['VRT', 'EEN'],
  'startTime': ['18:00:00','19:00:00'],
  'rLength': ['00:30:00', '00:45:00'],
  'live': [0,0]
})

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)

Ophalen kijkcijfer-data van 2025-04-22 00:00:00-2025-05-12 00:00:00
KijkcijferData opgehaald
Ophalen weer-data van 2025-04-22 00:00:00 tot 2025-05-13 00:00:00
Ophalen weer-data van 2025-04-22 00:00:00 tot 2025-05-13 00:00:00
Weerdata opgehaald


## Cleaning data

In [19]:
def cleanKijkcijferData(df):
    # Zet 'Kijkers' kolom, als 'rateInK' bestaat
    if 'rateInK' in df.columns:
        df['Kijkers'] = (
            df['rateInK']
            .dropna()
            .astype(str)
            .str.replace('.', '', regex=False)
            .astype(int)
        )
    else:
        df['Kijkers'] = None

    # Tijd aanpassen
    tijd_regex = r'^\d{2}:\d{2}:\d{2}$'
    # Omzetten naar datetime
    df['date'] = pd.to_datetime(df['dateDiff']).dt.date
    
    # Filter rijen met formaat
    df = df[df['startTime'].str.match(tijd_regex, na=False) & df['rLength'].str.match(tijd_regex, na=False)].copy()
    
    # Afleveringlengte naar seconden omzetten 
    df['Lengte_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds().astype(int)
    
    # Uren met 24+
    def time_cor(rij):
        tijdArr = rij['startTime'].split(':')
        if int(tijdArr[0]) >= 24:
            tijdArr[0] = str(int(tijdArr[0]) - 24).zfill(2)
            rij['date'] += timedelta(days=1)
        rij['startTime'] = ':'.join(tijdArr)
        return rij
    
    df = df.apply(time_cor, axis=1)

    # 1 kolom voor beide data
    df['FullDate'] = pd.to_datetime(df['date'].astype(str) 
                                    + " " + df['startTime'].astype(str))
    
    # Hour en minute voor join later on
    df['hour'] = pd.to_datetime(df['startTime'], format='%H:%M:%S').dt.hour
    df['minute'] = 0

    # Kolommen verwijderen die niet nodig meer zijn, als ze bestaan
    columns_to_drop = ['startTime', 'rLength', 'rateInK', 'ranking', 'live']
    df.drop([col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)

    # De nieuwe dataframe
    df = df[['FullDate', 'date', 'hour', 'minute', 'channel', 'description', 'Lengte_sec', 'Kijkers']]

    # Hernoemen kolommen
    df.rename(columns={'description': 'Programma', 'channel': 'Kanaal'}, inplace=True)

    # Nullwaarden verwijderen
    df.dropna(inplace=True)

    return df


def cleanWeerData(df):
  weerData = pd.read_csv('./data csv/weerdataraw.csv')
  weerData['timestamp'] = pd.to_datetime(weerData['timestamp'])
  #naar zelfde formaat als kijkcijfer datum
  weerData['datetime'] = weerData['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

  #hour voor join later on
  weerData['hour'] = pd.to_datetime(weerData['datetime']).dt.hour
  weerData['minute'] = pd.to_datetime(weerData['datetime']).dt.minute
  weerData['date'] = pd.to_datetime(weerData['datetime']).dt.date

  #verwijder kolom
  weerData = weerData.drop(columns=['timestamp'])

  weerData = weerData[['datetime', 'date' ,'hour', 'minute', 'temperature_2m', 'apparent_temperature', 
                            'rain', 'snowfall', 'weather_code', 'cloud_cover', 
                            'wind_speed_10m', 'sunshine_duration']]

  #hernoemen kolommen
  weerData.rename(columns={'temperature_2m':'Temperatuur', 'apparent_temperature':'Gevoelstemp', 'wind_speed_10m': 'Windsnelheid', 'rain':'Regen', 'snowfall': 'Sneeuw', 'weather_code':'Weercode', 'cloud_cover':'Bewolking', 'sunshine_duration':'Zonnenschijn'}, inplace=True)

  return weerData

def mergen(kijkcijfers, weer):
  print(kijkcijfers, weer)
  kijkcijfersWeer = pd.merge(kijkcijfers, weer, on=['date', 'hour'], how='left')
  kijkcijfersWeer = kijkcijfersWeer[['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec', 'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  kijkcijfersWeer.dropna(inplace=True)
  print(kijkcijfersWeer)
  return kijkcijfersWeer

In [20]:
histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
# Merge historische kijkcijfers en weerdata
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
#print("Aantal rijen in histKijkcijferWeerDf:", len(histKijkcijfersWeer))
#print(histKijkcijfersWeer.head()) 

               FullDate        date  hour  minute      Kanaal  \
0   2025-04-22 20:14:38  2025-04-22    20       0       VRT 1   
1   2025-04-22 19:00:04  2025-04-22    19       0       VRT 1   
2   2025-04-22 20:40:04  2025-04-22    20       0       VRT 1   
3   2025-04-22 19:47:18  2025-04-22    19       0       VRT 1   
4   2025-04-22 20:44:32  2025-04-22    20       0         VTM   
..                  ...         ...   ...     ...         ...   
415 2025-05-12 22:07:37  2025-05-12    22       0       VRT 1   
416 2025-05-12 19:59:02  2025-05-12    19       0       PLAY4   
417 2025-05-12 18:23:17  2025-05-12    18       0         VTM   
418 2025-05-12 20:00:03  2025-05-12    20       0  VRT CANVAS   
419 2025-05-12 21:25:53  2025-05-12    21       0  VRT CANVAS   

              Programma  Lengte_sec  Kijkers  
0                 THUIS        1374  1021385  
1    HET 7 UUR-JOURNAAL        2681   922137  
2       MET DE WIND MEE        2571   688346  
3      IEDEREEN BEROEMD        

## OneHotEncoding

In [28]:
def oneHot(df):
  with open('./models/oneHotEncoder.pkl', 'rb') as oneHotFile:
    oneHotEnc = pickle.load(oneHotFile)

  lageKard = df[[ 'hour','Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen']]
  dfOneHot = oneHotEnc.transform(lageKard)

  oneHotOutp = pd.DataFrame(dfOneHot.toarray(), 
                            columns=oneHotEnc.get_feature_names_out(), 
                            index=lageKard.index)

  df = df.drop(columns=['hour', 'Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen'])
  df = pd.concat([df, oneHotOutp], axis = 1)
  return df

## TargetEncoding

In [29]:
def target(df):
  #target encoding voor medium kardinaliteiten
  with open('./models/oneHotTarget.pkl', 'rb') as targetFile:
    targetEnc = pickle.load(targetFile) 
  medKardinaliteit = df[['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  #verdere feature engineering op vorig model
  target = targetEnc.fit_transform(medKardinaliteit, df['Kijkers'])
  df = df.drop(columns=['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn'])
  f = pd.concat([df, target], axis=1)

  return f


## Feature engineering

In [ ]:
def tijdFeatures(df):
    cleanData = df
    cleanData['date'] = pd.to_datetime(cleanData['date'])
    #feestdagen
    feestdagen = holidays.BE()
    cleanData['isFeestdag'] = cleanData['date'].apply(lambda x: 1 if x in feestdagen else 0)
    #dag van de week
    cleanData['Weekdag'] = cleanData['date'].dt.weekday
    #weekend
    cleanData['isWeekend'] = cleanData['Weekdag'].apply(lambda x: 1 if x >= 5 else 0)
    #seizoenen
    cleanData['Seizoen'] = cleanData['date'].apply(seizoenFinder)

    return cleanData

#seizoen
def seizoenFinder(datum):
    inputDatum = datum.date()
    Y = inputDatum.year
    seizoenen = {
        'lente': (date(Y, 3, 20), date(Y, 6, 20)),
        'zomer': (date(Y, 6, 21), date(Y, 9, 22)),
        'herfst':   (date(Y, 9, 23), date(Y, 12, 20)),
        'winter': (date(Y, 12, 21), date(Y + 1, 3, 19)),
    }

    for seizoen, (start, end) in seizoenen.items():
        if start <= inputDatum <= end:
            return seizoen
    return 'winter'

def createLag(df, n):
  for i in range(1,n+1):
    df[f'KijkersLag{i}'] = df.sort_values('FullDate').groupby('Programma')['Kijkers'].shift(i).ffill()
  return df

def lagFeatures(predDf, histKijkcijferWeerDf):
  predDf['Kijkers'] = np.nan

  for i in range(1, 4):
     predDf[f'KijkersLag{i}'] = predDf.sort_values('FullDate').groupby(['Programma'])['Kijkers'].shift(i)
     predDf[f'KijkersLag{i}'] = predDf[f'KijkersLag{i}'].fillna(predDf.groupby(['Programma'])['Kijkers'].transform('mean'))
     
  predDf = pd.concat([histKijkcijferWeerDf, predDf], ignore_index=True)
  return predDf   


## Pipeline

In [ ]:
def voorbereiding(df):
    # df['dateDiff'] = pd.to_datetime(df['dateDiff'])

    # end_date = df['dateDiff'].max()
    # start_date = end_date - timedelta(weeks=3)

    # # Haal historische data op
    # histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
    # histWeerdata = ophalenWeerData(start_date, end_date)
    # histWeerdataClean = cleanWeerData(histWeerdata)
    # histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)


    

    # Merge to predict data met weerdata
    toPredict = df.copy()
    toPredictClean = cleanKijkcijferData(toPredict)
    toPredictData = pd.merge(toPredictClean, histWeerdataClean, on=['date', 'hour'], how='left')

    # Drop rows met null waarden
    histKijkcijfersWeer.dropna(inplace=True)

    # Feature engineering op timestamp
    tijdFeatures(toPredictData)
    tijdFeatures(histKijkcijfersWeer)

    # Gebruik de functie in de make_prediction functie
    pred_hist_df = lagFeatures(toPredictData, histKijkcijfersWeer)

    # Haal de to_predict data er terug uit
    toPredictData = pred_hist_df[pred_hist_df['Kijkers'].isnull()]

    # One hot encoding voor programma
    toPredictData = oneHot(toPredictData)

    # Target encoding voor programma
    targetOneHotEnc = target(toPredictData)
    
    # Selecteer enkel numerieke kolommen
    toPredictNumeric = targetOneHotEnc.select_dtypes(include=[np.number])
    
    return toPredictNumeric

In [113]:
teVoorspellen = pd.DataFrame({
  'dateDiff' : ['2025-05-11 00:00:00.000000', '2025-05-11 00:00:00.000000'],
  'ranking': ['13', '234'],
  'description': ['THUIS', 'HET 7 UUR-JOURNAAL'],
  'channel': ['VRT', 'EEN'],
  'startTime': ['18:00:00','19:00:00'],
  'rLength': ['00:30:00', '00:45:00'],
  'live': [0,0]
})

data = voorbereiding(teVoorspellen)
print(data)

error: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
KijkcijferData opgehaald
Fetching weerdata van 2025-04-20 tot 2025-05-11
               FullDate        date  hour  minute      Kanaal  \
395 2025-05-10 12:34:27  2025-05-10    12       0       VRT 1   
396 2025-05-10 15:33:52  2025-05-10    15       0       VRT 1   
397 2025-05-10 16:54:52  2025-05-10    16       0       VRT 1   
398 2025-05-10 13:27:19  2025-05-10    13       0       VRT 1   
399 2025-05-10 20:14:50  2025-05-10    20       0  VRT CANVAS   

              Programma  Lengte_sec  Kijkers  
395            DE MARKT        1373   104013  
396  FLIKKEN MAASTRICHT        2868    94855  
397      RECHT OP RECHT        3603    93258  
398    VOLLEY. BB. (S.)        1203    90822  
399  TANKS EN TRACTOREN        1421    81572                     datetime        date  hour  minute  Temperatuur  \
74731  2025-04-10 19:00:00  2025-04-10   

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

## Voorspelling maken

In [ ]:
def voorspellingMaken(data):
  transformer = FunctionTransformer(data)

  voorbereidingPipeline = Pipeline([
        ('preprocess', transformer),
        ('std_scaler', StandardScaler())
    ])

  preprocessed_data = voorbereidingPipeline.fit_transform(data)

  with open('./models/lightGBM.pkl', 'rb') as file:
      xgb_model = pickle.load(file)

  predictions = xgb_model.predict(preprocessed_data)
  
  return predictions

prediction = voorspellingMaken(teVoorspellen)
prediction

In [ ]:
testData = pd.read_csv('./data csv/')